# SQLite에 대화 상태 영속화하기

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `SQLChatMessageHistory`는 메시지 행만 저장하는 방식입니다. 현재 권장되는
LangGraph checkpointer는 메시지를 포함한 **전체 그래프 상태**를 checkpoint로 저장하여
대화 재개, 상태 조회, fault tolerance 같은 기능을 함께 제공합니다.

이 예제는 `SqliteSaver`를 사용합니다. SQLite는 로컬 실습에 적합하고, 운영 환경에는
보통 Postgres 같은 서버형 checkpointer를 사용합니다.

참고: [LangGraph checkpointer 라이브러리](https://docs.langchain.com/oss/python/langgraph/checkpointers#checkpointer-libraries)


In [4]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" "langgraph-checkpoint-sqlite>=3.0" python-dotenv


In [5]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.6-luna")
model = init_chat_model(MODEL_ID)


## SQLite checkpointer 만들기

같은 파일과 `thread_id`를 사용하면 Python 프로세스를 다시 시작해도 상태를 복원할 수
있습니다. 처음부터 다시 실습하려면 연결을 만들기 전에 DB 파일을 삭제하세요.


In [6]:
import sqlite3
from pathlib import Path

from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH = Path("conversation_checkpoints.sqlite")

# 완전히 초기화하고 싶을 때만, DB 연결 전에 아래 줄의 주석을 해제하세요.
# DB_PATH.unlink(missing_ok=True)

connection = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="당신은 이전 대화를 기억하는 친절한 도우미입니다.",
    checkpointer=checkpointer,
)


## 한 thread에서 대화하기

`thread_id`가 기존 예제의 `session_id` 역할을 하며 checkpoint의 기본 조회 키가 됩니다.


In [7]:
config = {"configurable": {"thread_id": "user1:conversation1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕, 내 이름은 테디야."}]},
    config=config,
)
print(result["messages"][-1].content)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐라고?"}]},
    config=config,
)
print(result["messages"][-1].content)


안녕, 테디! 만나서 반가워 😊 무엇을 도와줄까?
네 이름은 **테디**야! 😊


## 연결을 닫고 새 agent에서 복원하기

아래 셀은 프로세스 재시작을 간단히 모사합니다. 기존 연결을 닫고 같은 SQLite 파일을
사용하는 새 checkpointer와 agent를 만듭니다.


In [8]:
connection.close()

restored_connection = sqlite3.connect(DB_PATH, check_same_thread=False)
restored_saver = SqliteSaver(restored_connection)
restored_saver.setup()

restored_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="당신은 이전 대화를 기억하는 친절한 도우미입니다.",
    checkpointer=restored_saver,
)

result = restored_agent.invoke(
    {"messages": [{"role": "user", "content": "제가 알려준 이름을 다시 말해 주세요."}]},
    config=config,
)
print(result["messages"][-1].content)


알려주신 이름은 **테디**입니다. 😊


같은 DB를 사용하더라도 다른 `thread_id`는 격리됩니다.


In [9]:
other_config = {"configurable": {"thread_id": "user1:conversation2"}}
result = restored_agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐라고?"}]},
    config=other_config,
)
print(result["messages"][-1].content)


아직 이름을 알려주지 않으셨어요. 원하시면 알려주시면 기억해둘게요!


In [10]:
# 노트북 작업이 끝나면 연결을 닫습니다.
restored_connection.close()
